# 00 — Data Download

Downloads both datasets to Google Drive for all downstream Colab notebooks.

| Dataset | GEO | Contents | Role |
|---------|-----|----------|------|
| Bhaduri et al. 2020, Nature | GSE132672 | Cortical organoids — 37 organoids, 3 protocols | Organoid data |
| Zhong et al. 2018, Nature | GSE104276 | Fetal PFC — gestational weeks 8–26 | Fetal brain reference |

**Run this notebook once.** After the data is on Drive, all other notebooks load from there.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Set Paths and Config

All paths are defined here. If you reorganize your Drive, only change this cell.

In [ ]:
import os

DRIVE_ROOT = '/content/drive/MyDrive/brain-organoid-trajectories'

DATASETS = {
    'bhaduri_2020': {
        'accession': 'GSE132672',
        'description': 'Bhaduri et al. 2020 — cortical organoids (3 protocols)',
    },
    'zhong_2018': {
        'accession': 'GSE104276',
        'description': 'Zhong et al. 2018 — fetal PFC atlas, GW8-26',
    },
}

for name, info in DATASETS.items():
    info['raw_dir']       = os.path.join(DRIVE_ROOT, 'data', 'raw', name)
    info['processed_dir'] = os.path.join(DRIVE_ROOT, 'data', 'processed', name)
    os.makedirs(info['raw_dir'], exist_ok=True)
    os.makedirs(info['processed_dir'], exist_ok=True)
    print(f"[{info['accession']}] {info['description']}")
    print(f"  raw:       {info['raw_dir']}")
    print(f"  processed: {info['processed_dir']}\n")

## 3. Verify GEO Accessions — List Available Files

In [ ]:
import ftplib

def list_geo_suppl(accession):
    prefix = accession[:len(accession)-3] + 'nnn'
    ftp_path = f'/geo/series/{prefix}/{accession}/suppl/'
    ftp = ftplib.FTP('ftp.ncbi.nlm.nih.gov')
    ftp.login()
    try:
        files = ftp.nlst(ftp_path)
        ftp.quit()
        return [f.split('/')[-1] for f in files]
    except ftplib.error_perm as e:
        ftp.quit()
        return [f'ERROR: {e}']

for name, info in DATASETS.items():
    acc = info['accession']
    print(f"--- {acc} ({info['description']}) ---")
    for f in list_geo_suppl(acc):
        print(f'  {f}')
    print()

## 4. Download Both Datasets

Skips files that already exist (safe to re-run).

In [ ]:
import ftplib
import os

def download_geo_suppl(accession, dest_dir):
    prefix = accession[:len(accession)-3] + 'nnn'
    ftp_path = f'/geo/series/{prefix}/{accession}/suppl/'
    ftp = ftplib.FTP('ftp.ncbi.nlm.nih.gov')
    ftp.login()
    files = ftp.nlst(ftp_path)
    print(f'  {len(files)} file(s) found')
    for filepath in files:
        filename = filepath.split('/')[-1]
        local_path = os.path.join(dest_dir, filename)
        if os.path.exists(local_path):
            size_mb = os.path.getsize(local_path) / 1e6
            print(f'  SKIP (exists, {size_mb:.1f} MB): {filename}')
            continue
        print(f'  Downloading: {filename} ...', end=' ', flush=True)
        with open(local_path, 'wb') as f:
            ftp.retrbinary(f'RETR {filepath}', f.write)
        size_mb = os.path.getsize(local_path) / 1e6
        print(f'done ({size_mb:.1f} MB)')
    ftp.quit()

for name, info in DATASETS.items():
    print(f"\n=== {info['accession']} — {name} ===")
    download_geo_suppl(info['accession'], info['raw_dir'])

print('\nAll downloads complete.')

## 5. Verify Downloads — List Files on Drive

In [ ]:
for name, info in DATASETS.items():
    raw_dir = info['raw_dir']
    print(f"\n--- {name} ({raw_dir}) ---")
    total_mb = 0
    for fname in sorted(os.listdir(raw_dir)):
        fpath = os.path.join(raw_dir, fname)
        size_mb = os.path.getsize(fpath) / 1e6
        total_mb += size_mb
        print(f'  {fname:<60} {size_mb:>8.1f} MB')
    print(f'  Total: {total_mb:.1f} MB')

## 6. Install Dependencies

In [ ]:
!pip install -q scanpy openpyxl xlrd

## 7. Load Bhaduri 2020 (Organoids)

File: `GSE132672_allorganoids_withnew_matrix.txt.gz`  
Format: gzipped tab-separated text, genes × cells — we transpose to cells × genes.  
Note: 1.5 GB file, loading takes a few minutes.

In [ ]:
import pandas as pd
import scanpy as sc
import anndata

bhaduri_file = os.path.join(DATASETS['bhaduri_2020']['raw_dir'],
                             'GSE132672_allorganoids_withnew_matrix.txt.gz')

print('Loading Bhaduri 2020 — this will take a few minutes...')
df = pd.read_csv(bhaduri_file, sep='\t', index_col=0, compression='gzip')
print(f'Raw matrix shape: {df.shape[0]:,} genes x {df.shape[1]:,} cells')

# Transpose: scanpy expects cells x genes
adata_organoid = anndata.AnnData(df.T)
print(f'AnnData shape:    {adata_organoid.shape[0]:,} cells x {adata_organoid.shape[1]:,} genes')
print(f'First 3 barcodes: {list(adata_organoid.obs_names[:3])}')
print(f'First 3 genes:    {list(adata_organoid.var_names[:3])}')

## 8. Load Zhong 2018 (Fetal PFC)

File: `GSE104276_all_pfc_2394_UMI_count_NOERCC.xls.gz`  
We use raw UMI counts (not TPM) so we can apply our own normalization.  
Despite the `.xls` extension, GEO files like this are typically tab-separated text.

In [ ]:
zhong_file = os.path.join(DATASETS['zhong_2018']['raw_dir'],
                           'GSE104276_all_pfc_2394_UMI_count_NOERCC.xls.gz')

print('Loading Zhong 2018...')

# GEO .xls files are usually tab-separated text despite the extension
try:
    df_fetal = pd.read_csv(zhong_file, sep='\t', index_col=0, compression='gzip')
    print('Loaded as tab-separated text.')
except Exception:
    # Fallback: try as actual Excel
    df_fetal = pd.read_excel(zhong_file, index_col=0)
    print('Loaded as Excel.')

print(f'Raw matrix shape: {df_fetal.shape[0]:,} genes x {df_fetal.shape[1]:,} cells')

# Transpose: scanpy expects cells x genes
adata_fetal = anndata.AnnData(df_fetal.T)
print(f'AnnData shape:    {adata_fetal.shape[0]:,} cells x {adata_fetal.shape[1]:,} genes')
print(f'First 3 barcodes: {list(adata_fetal.obs_names[:3])}')
print(f'First 3 genes:    {list(adata_fetal.var_names[:3])}')

## 9. Save as h5ad

Convert to `.h5ad` — scanpy's native format, fast to load in all downstream notebooks.  
No filtering or normalization applied — this is the raw object.

In [ ]:
def save_h5ad(adata, processed_dir, filename):
    out_path = os.path.join(processed_dir, filename)
    adata.write_h5ad(out_path)
    size_mb = os.path.getsize(out_path) / 1e6
    print(f'Saved: {out_path} ({size_mb:.1f} MB)')

save_h5ad(adata_organoid, DATASETS['bhaduri_2020']['processed_dir'], 'bhaduri_2020_raw.h5ad')
save_h5ad(adata_fetal,    DATASETS['zhong_2018']['processed_dir'],   'zhong_2018_raw.h5ad')

## Done

Data is now on Google Drive:
- `data/raw/bhaduri_2020/` — original GEO files
- `data/raw/zhong_2018/` — original GEO files
- `data/processed/bhaduri_2020/bhaduri_2020_raw.h5ad`
- `data/processed/zhong_2018/zhong_2018_raw.h5ad`

Next: `colab_01_preprocessing.ipynb` — QC, filtering, normalization, HVG, PCA on both datasets.